In [ ]:
%run ./00_Setup_Mount

# =============================================================
# 05_BATCH_INFERENCE.IPYNB (FINAL VERSION)
# =============================================================



In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import sys

# Agregar src al path
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.ml.scorer import score_dataframe

GOLD_PATH = f"s3a://{BUCKET}/gold/app_inmuebles/"
SCORED_PATH = f"s3a://{BUCKET}/gold/app_inmuebles_scored/"
MODEL_DIR = "/dbfs/mnt/models/champion"

print(f"📖 Cargando Parquet desde: {GOLD_PATH}")

# Cargar con opciones explícitas (Evita usar spark.conf.set que está bloqueado)
# Usando format("parquet") para máxima compatibilidad con .options()
df_gold_spark = spark.read.format("parquet").options(**S3_OPTIONS).load(GOLD_PATH)

print("⬇️ Bajando a Pandas (Acción de Spark)...")
df_pandas = df_gold_spark.toPandas()
print(f"✅ {len(df_pandas)} registros cargados.")


In [ ]:
import boto3
import io
import re
from datetime import datetime

print(f"🔍 Buscando el último modelo en S3: s3://{BUCKET}/models/ ...")

s3_client = boto3.client(
    's3',
    aws_access_key_id=S3_OPTIONS.get('fs.s3a.access.key'),
    aws_secret_access_key=S3_OPTIONS.get('fs.s3a.secret.key'),
    region_name='us-east-1'
)

try:
    response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='models/')
    if 'Contents' not in response:
        raise FileNotFoundError(f"No se encontraron archivos en s3://{BUCKET}/models/")
    
    # Filtrar archivos .pkl que contengan 'bundle_v'
    model_files = [
        obj for obj in response['Contents'] 
        if obj['Key'].endswith('.pkl') and 'bundle_v' in obj['Key']
    ]
    
    if not model_files:
        raise FileNotFoundError("No se encontró ningún bundle de modelo (.pkl) en la ruta de S3.")
    
    # Ordenar por fecha de última modificación en S3
    latest_model_obj = max(model_files, key=lambda x: x['LastModified'])
    model_key = latest_model_obj['Key']
    
    print(f"📦 Descargando modelo Champion desde S3: {model_key} (Subido el: {latest_model_obj['LastModified']})")
    
    model_response = s3_client.get_object(Bucket=BUCKET, Key=model_key)
    bundle = pickle.load(io.BytesIO(model_response['Body'].read()))
    
    print(f"✅ Modelo {bundle.get('model_version', 'v?')} cargado exitosamente desde S3.")
except Exception as e:
    print(f"❌ Error crítico cargando el modelo desde S3: {e}")
    raise e


In [ ]:
print("⚙️ Generando rentabilidad...")
df_scored_pandas = score_dataframe(df_pandas, bundle)
print("✅ Completado.")
display(df_scored_pandas[["titulo", "city_token", "precio_num", "precio_predicho", "rentabilidad_potencial", "estado_inversion"]].head(10))


In [ ]:
print("⬆️ Guardando resultado...")

# Limpieza de tipos
for col in df_scored_pandas.columns:
    if df_scored_pandas[col].dtype == "object":
        df_scored_pandas[col] = df_scored_pandas[col].fillna("").astype(str)

df_scored_spark = spark.createDataFrame(df_scored_pandas)

# Guardar como Delta
print(f"💾 Guardando en: {SCORED_PATH}")
writer = df_scored_spark.write.format("delta").mode("overwrite").options(**S3_OPTIONS).option("overwriteSchema", "true")
writer.save(SCORED_PATH)

print("🎉 Finalizado con éxito.")
